# Experiments with a Simple ResNet on CIFAR-10

This notebook contains various experiments and analyses using the custom **ResNet** implementation defined in `src/resnet_keras.py`.

## Table of Contents

1. Brief description of the architecture.
2. Loading the model and trained weights.
3. Evaluation on the test set.
4. Visualization of metrics and training curves.
5. Inspection of predictions and error analysis.

## Imports

In [2]:
import tensorflow as tf
from pathlib import Path
import sys
import os
from google.colab import drive

drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/ResNets")
os.chdir(PROJECT_ROOT)

sys.path.append(str(PROJECT_ROOT / "src"))

from resnet_keras import build_resnet50
DATA_DIR = PROJECT_ROOT / "data"

print("TensorFlow version:", tf.__version__)

MessageError: Error: credential propagation was unsuccessful

## Training visualization functions

In [ ]:
!pip install livelossplot

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from livelossplot import PlotLossesKeras

class RealTimeVisualizer(tf.keras.callbacks.Callback):
    def __init__(self, validation_ds):
        super().__init__()
        self.val_images, self.val_labels = next(iter(validation_ds.take(1)))
        self.class_names = ['avión', 'auto', 'pájaro', 'gato', 'ciervo',
                            'perro', 'rana', 'caballo', 'barco', 'camión']

    def on_epoch_end(self, epoch, logs=None):
        preds = self.model.predict(self.val_images, verbose=0)

        plt.figure(figsize=(15, 3))
        for i in range(5):
            plt.subplot(1, 5, i+1)
            img = self.val_images[i].numpy()
            if img.max() <= 1.0: img = (img * 255).astype(np.uint8)

            plt.imshow(img.astype(np.uint8))

            p_idx, r_idx = np.argmax(preds[i]), np.argmax(self.val_labels[i])
            color = 'green' if p_idx == r_idx else 'red'

            plt.title(f"P: {self.class_names[p_idx]}\nR: {self.class_names[r_idx]}", color=color)
            plt.axis('off')
        plt.show()

## Load weights from path

In [2]:
model = build_resnet50(input_shape=(160, 160, 3), classes=10)

weights_path = PROJECT_ROOT / "checkpoints/resnet_best.keras"

model = tf.keras.models.load_model(weights_path)

print(f"Model loaded from: {weights_path}")

Model loaded from: /content/drive/MyDrive/Colab Notebooks/ResNets/checkpoints/resnet_best.keras


## Train with dataset CIFAR-10

In [ ]:
from train_keras import train_with_cifar, CONFIG, get_cifar10_dataset


CONFIG["epochs"] = 20
CONFIG["learning_rate"] = 0.01
CONFIG["batch_size"] = 64

# _, val_ds = get_cifar10_dataset(
#     batch_size=CONFIG["batch_size"],
#     target_size=CONFIG["target_size"]
# )

# visualizadores = [
#     PlotLossesKeras(),
# ]


history, model = train_with_cifar(
    config=CONFIG,
    project_path=str(PROJECT_ROOT)
)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
Initializing new model with aggressive augmentation...
Epoch 1/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 0s 562ms/step - accuracy: 0.2626 - loss: 3.3451
Epoch 1: val_accuracy improved from None to 0.45690, saving model to /content/drive/MyDrive/Colab Notebooks/ResNets/checkpoints/resnet_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Colab Notebooks/ResNets/checkpoints/resnet_best.keras
782/782 ━━━━━━━━━━━━━━━━━━━━ 515s 607ms/step - accuracy: 0.3346 - loss: 2.1328 - val_accuracy: 0.4569 - val_loss: 1.4973 - learning_rate: 0.0100
Epoch 2/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 0s 574ms/step - accuracy: 0.4270 - loss: 1.5842
Epoch 2: val_accuracy improved from 0.45690 to 0.50960, saving model to /content/drive/MyDrive/Colab Notebooks/ResNets/checkpoints/resnet_best.keras

Epoch 2: finished saving model to /content/drive/MyDrive/Colab Notebooks/ResNets/checkpoints/resnet_best.keras
782/782 ━━━━━━━━━━━━━━━━━━━━ 473s 605ms/step - accur

## Resume training from loaded model

In [ ]:
from tensorflow.keras.models import load_model
from train_keras import train_with_cifar, CONFIG, get_cifar10_dataset

weights_path = "/content/drive/MyDrive/Colab Notebooks/ResNets/checkpoints/resnet_best.keras"
model = load_model(weights_path)

CONFIG["epochs"] = 40
CONFIG["initial_epoch"] = 20
CONFIG["learning_rate"] = 0.01
CONFIG["batch_size"] = 64

# _, val_ds = get_cifar10_dataset(
#     batch_size=CONFIG["batch_size"],
#     target_size=CONFIG["target_size"]
# )

# visualizadores = [
#     PlotLossesKeras(),
# ]


history, model = train_with_cifar(
    model=model,
    config=CONFIG,
    project_path=str(PROJECT_ROOT)
)

Epoch 21/40
782/782 ━━━━━━━━━━━━━━━━━━━━ 0s 568ms/step - accuracy: 0.8107 - loss: 0.5403
Epoch 21: val_accuracy improved from None to 0.75490, saving model to /content/drive/MyDrive/Colab Notebooks/ResNets/checkpoints/resnet_best.keras

Epoch 21: finished saving model to /content/drive/MyDrive/Colab Notebooks/ResNets/checkpoints/resnet_best.keras
782/782 ━━━━━━━━━━━━━━━━━━━━ 495s 603ms/step - accuracy: 0.8092 - loss: 0.5452 - val_accuracy: 0.7549 - val_loss: 0.7292 - learning_rate: 0.0100
Epoch 22/40
782/782 ━━━━━━━━━━━━━━━━━━━━ 0s 569ms/step - accuracy: 0.8159 - loss: 0.5279
Epoch 22: val_accuracy improved from 0.75490 to 0.80770, saving model to /content/drive/MyDrive/Colab Notebooks/ResNets/checkpoints/resnet_best.keras

Epoch 22: finished saving model to /content/drive/MyDrive/Colab Notebooks/ResNets/checkpoints/resnet_best.keras
782/782 ━━━━━━━━━━━━━━━━━━━━ 470s 601ms/step - accuracy: 0.8172 - loss: 0.5233 - val_accuracy: 0.8077 - val_loss: 0.5704 - learning_rate: 0.0100
Epoch 23/

## Test Evaluation

In [3]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

x_test = x_test[:500]
y_test = y_test[:500]

x_test = x_test.astype("float32") / 255.0
x_test = tf.image.resize(x_test, (160, 160))

# One-hot encoding de las etiquetas
num_classes = 10
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes)

# eval
test_loss, test_acc = model.evaluate(x_test, y_test_cat, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 66s 0us/step
Test loss: 0.3550
Test accuracy: 0.8940


## Test with an image

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf


img_path = '/content/drive/MyDrive/Colab Notebooks/ResNets/Villa.jpg'
img = tf.keras.preprocessing.image.load_img(img_path, target_size=(160, 160))

img_array = tf.keras.preprocessing.image.img_to_array(img)
img_array = img_array / 255.0
input_img = np.expand_dims(img_array, axis=0)

predictions = model.predict(input_img)
predicted_class = np.argmax(predictions)
confidence = np.max(predictions)

classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

plt.imshow(img)
plt.title(f"Prediction: {classes[predicted_class]} ({confidence*100:.2f}% confidence)")
plt.axis('off')
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Colab Notebooks/ResNets/Villa.jpg'

In [38]:
for i, conf in enumerate(predictions[0]):
    print(f"{classes[i]}: {conf*100:.2f}%")

avión: 0.41%
automóvil: 5.84%
pájaro: 0.10%
gato: 58.20%
ciervo: 0.00%
perro: 0.01%
rana: 2.94%
caballo: 3.03%
barco: 0.17%
camión: 29.30%
